# WARNING: ARCHIVAL - NOT RUNNABLE IN THIS REPO

이 노트북은 원본 로컬 환경에서 진행된 **데이터 병합 과거 작업 기록**입니다.
각 셀은 OneDrive 바탕화면 등의 **절대경로**를 사용하며, 그 경로에 있던 중간 산출물들은 새 저장소 레이아웃(KBO-pitcher-fatigue/data/...)에 존재하지 않습니다.

**용도:** 어떤 컬럼이 어떤 원천에서 어떤 순서로 머지되어 최종 fatigue_with_index.csv가 만들어졌는지를 추적하기 위한 참조용입니다.

**재실행하려면:** 각 절대경로를 새 레포의 상응하는 파일(../data/raw/..., ../data/processed/...)로 교체해야 합니다.

In [1]:
import pandas as pd
from pathlib import Path

# 1) CSV 파일들이 들어있는 디렉터리 경로
data_dir = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\구사율")

# 2) “.csv”로 끝나는 파일만 리스트업
csv_files = list(data_dir.glob("*.csv"))

print(f"Found {len(csv_files)} files:" , [f.name for f in csv_files])

df_list = []
for f in csv_files:
    try:
        # 우선 UTF-8로 읽어보기
        df = pd.read_csv(f, encoding='utf-8')
    except UnicodeDecodeError:
        # 실패하면 CP949로 다시 읽기
        df = pd.read_csv(f, encoding='cp949')
    df_list.append(df)

# 합치기
merged_df = pd.concat(df_list, ignore_index=True)

# 저장
merged_df.to_csv(data_dir / "merged_구속.csv", index=False, encoding='utf-8-sig')

print(f"Merged {len(df_list)} files into merged_구속.csv")

Found 10 files: ['KIA.csv', 'KT.csv', 'LG.csv', 'NC.csv', 'SSG.csv', '두산.csv', '롯데.csv', '삼성.csv', '키움.csv', '한화.csv']
Merged 10 files into merged_구속.csv


In [9]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정
file_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\2025년 mykbo.csv")

# 2) CSV 읽기 (한글 윈도우 기본은 cp949)
df = pd.read_csv(file_path, encoding='utf-8')

# 3) Date 컬럼을 datetime으로 변환 및 선수·연도·날짜로 정렬
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Name', 'Year', 'Date']).reset_index(drop=True)

# 4) 선수·시즌별 등판 간 일수(diff) 계산
df['day_diff'] = df.groupby(['Name', 'Year'])['Date'].diff().dt.days

# 5) 연투 여부, 휴식일 수 추가
df['연투 여부'] = df['day_diff'] == 1
df['휴식일 수'] = (df['day_diff'] - 1).fillna(0).astype(int)

# 6) 선수·시즌별 연투일 누적 계산
df['연투일'] = 0
for (name, year), grp in df.groupby(['Name', 'Year']):
    cnt = 1
    for idx in grp.index:
        if df.at[idx, 'day_diff'] == 1:
            cnt += 1
        else:
            cnt = 1
        df.at[idx, '연투일'] = cnt

# 7) 중간 컬럼 제거
df.drop(columns=['day_diff'], inplace=True)

# 8) 결과를 새로운 CSV로 저장 (utf-8-sig 인코딩으로 한글 깨짐 방지)
output_path = file_path.parent / 'merged_날씨_with_fatigue.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔️ 계산된 데이터를 '{output_path}'에 저장했습니다.")

✔️ 계산된 데이터를 'C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\merged_날씨_with_fatigue.csv'에 저장했습니다.


In [ ]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정
fatigue_path       = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\merged_날씨_with_fatigue.csv")
participation_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\국대.csv")

# 2) CSV 읽기
df_fatigue = pd.read_csv(fatigue_path, encoding='utf-8-sig')
df_part    = pd.read_csv(participation_path, encoding='cp949')

# 3) 이벤트 컬럼 리스트 추출 (Name, Year 제외)
event_cols = [c for c in df_part.columns if c not in ('Name','Year')]

# 4) 선수별 국대 참여 집계: 어떤 연도에든 TRUE면 전체 TRUE
df_part_agg = (
    df_part
    .groupby('Name')[event_cols]
    .any()
    .reset_index()
)

# 5) Name 기준으로 병합 (Year 무시)
df_merged = pd.merge(
    df_fatigue,
    df_part_agg,
    on='Name',
    how='left'
)

# 6) NaN → False 처리 (이제 event_cols에만 있는 컬럼이므로 KeyError 없음)
df_merged[event_cols] = df_merged[event_cols].fillna(False)

# 7) 결과 저장
output_path = fatigue_path.parent / 'merged_날씨_최종.csv'
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔ 저장 완료: {output_path.name}")

✔ 저장 완료: merged_날씨_최종.csv


C:\Users\yun72_92xubzr\AppData\Local\Temp\ipykernel_23388\3335580956.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_merged[event_cols] = df_merged[event_cols].fillna(False)


In [12]:
from pathlib import Path
import pandas as pd
import unicodedata

# 1) 파일 경로 설정 (실제 경로로 수정)
file_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\merged_날씨_with_fatigue.csv")

# 2) CSV 읽기 (utf-8-sig 인코딩)
df = pd.read_csv(file_path, encoding='utf-8-sig')

# 3) IP 문자열 → float 변환 함수 정의
def ip_to_float(x):
    if pd.isna(x):
        return float('nan')
    total = 0.0
    for part in str(x).split():
        try:
            # 정수나 소수
            total += float(part)
        except ValueError:
            # 유니코드 분수(⅓, ½ 등)
            total += unicodedata.numeric(part)
    return total

# 4) IP 컬럼에 함수 적용 (기존 IP 컬럼을 덮어씁니다)
df['IP'] = df['IP'].apply(ip_to_float)

# 5) 수정된 데이터 저장
df.to_csv(file_path, index=False, encoding='utf-8-sig')

print("✔ IP 컬럼이 float 타입으로 변환되어 저장되었습니다.")

✔ IP 컬럼이 float 타입으로 변환되어 저장되었습니다.


In [14]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정
file_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\merged_날씨_with_fatigue.csv")

# 2) CSV 읽기
df = pd.read_csv(file_path, encoding='utf-8-sig')

# 3) Team 값 매핑 사전
team_mapping = {
    'NC':        'NC',
    'Lotte':     '롯데',
    'SSG':       'SSG',
    'Kiwoom':    '키움',
    'Kia':       'KIA',
    'LG':        'LG',
    'Doosan':    '두산',
    'Samsung':   '삼성',
    'Hanwha':    '한화',
    'KT':        'KT'
}

# 4) Team 컬럼 치환
df['Team'] = df['Team'].map(team_mapping).fillna(df['Team'])

# 5) 파일 저장
output_path = file_path.parent / 'merged_최종(2).csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔ Team 컬럼이 매핑 및 재정렬되어 '{output_path.name}' 으로 저장되었습니다.")

✔ Team 컬럼이 매핑 및 재정렬되어 'merged_최종(2).csv' 으로 저장되었습니다.


In [ ]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정
merged_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_최종.csv")
dist_path   = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\거리.csv")

# 2) CSV 읽기
df          = pd.read_csv(merged_path, encoding='utf-8-sig')
df_dist     = pd.read_csv(dist_path,   encoding='utf-8-sig')

# 3) 병합 (Team, Year 기준)
df_merged = pd.merge(
    df,
    df_dist,
    left_on = ['Team','Date'],
    right_on= ['구단','날짜_정렬'],
    how      = 'left'
)

# 4) 불필요 컬럼 제거
df_merged.drop(columns=['구단','날짜_정렬'], inplace=True)

# 5) 결과 저장
output_path = merged_path.parent / 'merged_이동거리포함.csv'
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔ 저장 완료: {output_path.name}")

✔ 저장 완료: merged_날씨_최종_이동거리포함.csv


In [12]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정 (실제 경로로 수정)
merged_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_최종.csv")
usage_path  = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_구사율, 피안타율.csv")

# 2) CSV 읽기
df_merged = pd.read_csv(merged_path, encoding='utf-8-sig')
df_usage  = pd.read_csv(usage_path,  encoding='cp949')  # 한글 포함 파일이라면 cp949

# 3) Name, Year 기준으로 구사율 붙이기
df_final = pd.merge(
    df_merged,
    df_usage,
    on=['Name','Year'],
    how='left'
)

# 4) 결과 저장
output_path = merged_path.parent / 'merged_최종_구사율포함.csv'
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔ 저장 완료: {output_path.name}")

✔ 저장 완료: merged_최종_구사율포함.csv


In [16]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정
merged_path    = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_최종.csv")
sspitcher_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\ss_pitcher_with_temp.csv")

# 2) CSV 읽기 (인코딩은 환경에 맞게 조정)
df_main = pd.read_csv(merged_path, encoding='cp949')
df_ss   = pd.read_csv(sspitcher_path, encoding='cp949')

# 3) Date 컬럼을 datetime으로 변환
df_main['Date'] = pd.to_datetime(df_main['Date'])
df_ss  ['Date'] = pd.to_datetime(df_ss ['Date'], errors='coerce')

# 4) ss 데이터에서 Stadium·Temp만 추출
df_ss_sub = df_ss[['Name','Date','Stadium','Temp']]

# 5) 병합하여 _new 컬럼 생성
df_merged = pd.merge(
    df_main,
    df_ss_sub,
    on=['Name','Date'],
    how='left',
    suffixes=('','_new')
)

# 6) 이승현_좌/이승현_우인 경우에만 빈 값(또는 NaN) 채우기
mask = df_merged['Name'].isin(['이승현_좌','이승현_우'])

# 원본 Stadium이 비어 있거나 NaN일 때만 new 값으로 대체
df_merged.loc[mask, 'Stadium'] = df_merged.loc[mask, 'Stadium'].fillna(df_merged.loc[mask, 'Stadium_new'])
df_merged.loc[mask, 'Temp']    = df_merged.loc[mask, 'Temp'].   fillna(df_merged.loc[mask, 'Temp_new'])

# 7) 임시 컬럼 제거
df_merged.drop(columns=['Stadium_new','Temp_new'], inplace=True)

# 8) 결과 저장
output_path = merged_path.parent / 'merged_최종_이승현_보완.csv'
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔ 완료: {output_path.name} 에 이승현_좌/우 Stadium·Temp 보완 저장")

✔ 완료: merged_최종_이승현_보완.csv 에 이승현_좌/우 Stadium·Temp 보완 저장


In [15]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정 (실제 경로로 수정)
merged_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\merged_최종(2).csv")
bio_path    = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\2025년 바이오.csv")  # Name,Height,Weight,BirthYear,Age

# 2) CSV 읽기 (인코딩은 환경에 맞게 조정)
df_main = pd.read_csv(merged_path)
df_bio  = pd.read_csv(bio_path, encoding='cp949')

# 3) Name 기준으로 병합
df_merged = pd.merge(
    df_main,
    df_bio,
    on='Name',
    how='left'
)

# 4) 결과 저장
output_path = merged_path.parent / 'merged_최종_신체정보포함.csv'
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔ 바이오 정보 병합 완료: {output_path.name}")

✔ 바이오 정보 병합 완료: merged_최종_신체정보포함.csv


In [45]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정
merged_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_선수기록.csv")
speed_path  = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_평균구속.csv")

# 2) CSV 읽기 (인코딩은 파일에 맞게 조정)
df_main  = pd.read_csv(merged_path, encoding='cp949')
df_speed = pd.read_csv(speed_path)

# 3) Name 열 공백 제거
df_main['Name']  = df_main['Name'].str.strip()
df_speed['Name'] = df_speed['Name'].str.strip()

# 4) Date 컬럼을 다양한 포맷 인식하도록 datetime 변환
df_main['Date']  = pd.to_datetime(df_main['Date'],  errors='coerce', infer_datetime_format=True)
df_speed['Date'] = pd.to_datetime(df_speed['Date'], errors='coerce', infer_datetime_format=True)

# 5) 병합할 속도 컬럼 리스트
speed_cols = [
    '전체구속','2Seam','4Seam','Cutter','Curve',
    'Slider','Changeup','Sinker','Forkball','Knuckle','Other'
]

# 6) Name+Date 기준으로 병합
df_merged = pd.merge(
    df_main,
    df_speed[['Name','Date'] + speed_cols],
    on=['Name','Date'],
    how='left'
)

# 7) 매칭 실패 확인
mismatch = (
    df_main[['Name','Date']]
    .drop_duplicates()
    .merge(df_speed[['Name','Date']], on=['Name','Date'], how='left', indicator=True)
)
not_matched = mismatch[mismatch['_merge']=='left_only'].drop(columns=['_merge'])

# 8) 매칭 실패만 별도 CSV로 저장
unmatched_path = merged_path.parent / '매칭실패_선수_날짜.csv'
not_matched.to_csv(unmatched_path, index=False, encoding='utf-8-sig')
print(f"▶ 매칭 실패 조합 {len(not_matched)}건을 '{unmatched_path.name}'에 저장했습니다.")

# 9) 전체 병합 결과도 저장
output_path = merged_path.parent / 'merged_선수기록_with_speeds.csv'
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✔ 전체 병합 결과를 '{output_path.name}'에 저장했습니다.")

C:\Users\yun72_92xubzr\AppData\Local\Temp\ipykernel_18552\1194536194.py:17: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_main['Date']  = pd.to_datetime(df_main['Date'],  errors='coerce', infer_datetime_format=True)
C:\Users\yun72_92xubzr\AppData\Local\Temp\ipykernel_18552\1194536194.py:18: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_speed['Date'] = pd.to_datetime(df_speed['Date'], errors='coerce', infer_datetime_format=True)


▶ 매칭 실패 조합 269건을 '매칭실패_선수_날짜.csv'에 저장했습니다.
✔ 전체 병합 결과를 'merged_선수기록_with_speeds.csv'에 저장했습니다.


In [36]:
from pathlib import Path
import pandas as pd
from dateutil import parser

# 1) 파일 경로
input_path  = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_평균구속.csv")
output_path = input_path.parent / 'merged_평균구속_날짜통일.csv'

# 2) 읽기
df = pd.read_csv(input_path, encoding='cp949')

# 3) 원본 보관
df['Date_orig'] = df['Date'].astype(str)

# 4) 구분자 통일: '.' → '-'
df['Date_clean'] = df['Date_orig'].str.replace(r'\.', '-', regex=True)

# 5) 파싱 함수: 실패 시 dateutil.parser 로도 시도
def try_parse(x):
    try:
        # 첫 시도: pandas가 infer_datetime_format 으로 자동 파싱
        return pd.to_datetime(x, errors='raise', infer_datetime_format=True)
    except:
        try:
            # 두 번째: dateutil 의 범용 파서 사용
            return parser.parse(x)
        except:
            return pd.NaT

# 6) 파싱 적용
df['Date_parsed'] = df['Date_clean'].apply(try_parse)

# 7) 여전히 NaT인 행 확인 (필요시 로깅)
missing = df[df['Date_parsed'].isna()]
if not missing.empty:
    print("파싱 실패한 원본 날짜들:")
    print(missing['Date_orig'].unique())

# 8) 최종 포맷으로 변환
df['Date'] = df['Date_parsed'].dt.strftime('%Y-%m-%d')

# 9) 불필요 컬럼 제거
df = df.drop(columns=['Date_orig','Date_clean','Date_parsed'])

# 10) 저장
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✔ 저장 완료: {output_path.name}")

C:\Users\yun72_92xubzr\AppData\Local\Temp\ipykernel_18552\4036889950.py:22: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(x, errors='raise', infer_datetime_format=True)


✔ 저장 완료: merged_평균구속_날짜통일.csv


In [7]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정
main_path   = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_최종.csv")
speed2_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_평균구속.csv")

# 2) CSV 읽기
df_main   = pd.read_csv(main_path,   encoding='cp949')
df_speed2 = pd.read_csv(speed2_path, encoding='cp949')

# 3) Date 컬럼 datetime 변환
df_main['Date']   = pd.to_datetime(df_main['Date'],   errors='coerce', infer_datetime_format=True)
df_speed2['Date'] = pd.to_datetime(df_speed2['Date'], errors='coerce', infer_datetime_format=True)

# 4) 추가할 속도 컬럼 리스트
cols = [
    '전체구속','2Seam','4Seam','Cutter','Curve',
    'Slider','Changeup','Sinker','Forkball','Knuckle','Other'
]

# 5) 병합: df_main 에 df_speed2의 속도 컬럼을 그대로 추가
df_final = pd.merge(
    df_main,
    df_speed2[['Name','Date'] + cols],
    on=['Name','Date'],
    how='left'
)

# 6) 중복 행 제거: 동일한 모든 컬럼 값이 중복된 경우 첫 번째 행만 남기고 제거
df_final = df_final.drop_duplicates(keep='first')

# 7) 결과 저장
output_path = main_path.parent / 'merged_최종_with_speed_deduped.csv'
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔ 중복 행 제거 후 저장 완료: {output_path.name}")

C:\Users\yun72_92xubzr\AppData\Local\Temp\ipykernel_10016\3108492455.py:13: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_main['Date']   = pd.to_datetime(df_main['Date'],   errors='coerce', infer_datetime_format=True)
C:\Users\yun72_92xubzr\AppData\Local\Temp\ipykernel_10016\3108492455.py:14: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_speed2['Date'] = pd.to_datetime(df_speed2['Date'], errors='coerce', infer_datetime_format=True)


✔ 중복 행 제거 후 저장 완료: merged_최종_with_speed_deduped.csv


In [ ]:
import pandas as pd
from pathlib import Path

# 1) 파일 경로
output_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\데이터\merged_최종_with_speed_deduped.csv")

# 2) 파일 읽기 (Date 컬럼은 datetime으로 파싱)
df = pd.read_csv(output_path, encoding='utf-8-sig', parse_dates=['Date'])

# 3) Name–Date별 개수 집계
dup_counts = (
    df
    .groupby(['Name','Date'])
    .size()
    .reset_index(name='count')
)

# 4) 중복(2개 이상)만 필터
dup = dup_counts[dup_counts['count'] > 1]

# 5) 결과 출력
if dup.empty:
    print("✅ Name–Date 조합 중복 없음")
else:
    print("⚠️ 중복된 Name–Date 조합 목록:")
    print(dup)

✅ Name–Date 조합 중복 없음


In [16]:
from pathlib import Path
import pandas as pd

# 1) 파일 경로 설정 (실제 위치로 수정)
input_path  = Path(r"C:\Users\yun72_92xubzr\Downloads\fatigue_df.csv")
output_path = Path(r"C:\Users\yun72_92xubzr\Downloads\fatigue_df_with_fastball.csv")

# 2) CSV 읽기
df = pd.read_csv(input_path, encoding='utf-8-sig')

# 3) FastballVelo 계산: 2Seam, 4Seam 둘 다 있으면 평균, 하나만 있으면 그 값
df['속도'] = df[['2Seam', '4Seam']].mean(axis=1, skipna=True)

# 4) 나머지 속도 컬럼 제거
velocity_cols = [
    '전체구속','2Seam','4Seam','Cutter','Curve',
    'Slider','Changeup','Sinker','Forkball','Knuckle','Other'
]
df = df.drop(columns=[c for c in velocity_cols if c in df.columns])

# 5) 수정된 DataFrame을 CSV로 저장
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✔ 저장 완료: {output_path}")

✔ 저장 완료: C:\Users\yun72_92xubzr\Downloads\fatigue_df_with_fastball.csv


In [12]:
import pandas as pd
from pathlib import Path

# 1) 파일 경로 설정
input_path  = Path(r"C:\Users\yun72_92xubzr\Downloads\fatigue_df_직구평속.csv")
output_path = input_path.parent / 'fatigue_df_with_fip_delta.csv'

# 2) CSV 읽기
df = pd.read_csv(input_path, encoding='utf-8-sig', parse_dates=['Date'])

# 3) 원본 FIP 결측 마스크
orig_nan = df['FIP'].isna()

# 4) 정렬
df = df.sort_values(['Name','Year','Date'])

# 5) FIP 채우기 & ΔFIP 계산
df['FIP_filled'] = df.groupby(['Name','Year'])['FIP'].transform(lambda s: s.ffill())
df['delta_FIP']  = df.groupby(['Name','Year'])['FIP_filled'].transform(lambda s: s.diff())

# 6) 원본 FIP 결측 행은 Δ도 NaN
df.loc[orig_nan, 'delta_FIP'] = pd.NA

# 7) 중복 열 제거
df = df.loc[:, ~df.columns.duplicated()]

# 8) 컬럼 순서 재배치: FIP 바로 다음에 delta_FIP
cols = df.columns.tolist()
# FIP_filled 제거
if 'FIP_filled' in cols:
    cols.remove('FIP_filled')

# delta_FIP만 FIP 뒤로 이동
cols.remove('delta_FIP')
idx = cols.index('FIP') + 1
cols.insert(idx, 'delta_FIP')

df = df[cols]

# 9) 결과 확인
print(df[['Name','Year','Date','FIP','delta_FIP']].head(10))

# 10) 저장
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✔ 저장 완료: {output_path}")

  Name  Year       Date    FIP  delta_FIP
0  고영표  2021 2021-04-07  4.003        NaN
1  고영표  2021 2021-04-13  1.337     -2.666
2  고영표  2021 2021-04-18  3.670      2.333
3  고영표  2021 2021-04-24  2.503     -1.167
4  고영표  2021 2021-04-30  3.599      1.096
5  고영표  2021 2021-05-06  3.670      0.071
6  고영표  2021 2021-05-12  5.337      1.667
7  고영표  2021 2021-05-26  3.170     -2.167
8  고영표  2021 2021-06-01  2.720     -0.450
9  고영표  2021 2021-06-08  3.003      0.283
✔ 저장 완료: C:\Users\yun72_92xubzr\Downloads\fatigue_df_with_fip_delta.csv


In [22]:
import pandas as pd
from pathlib import Path

# 1) 파일 경로 설정 — 실제 위치로 수정하세요
input_path  = Path(r"C:\Users\yun72_92xubzr\Downloads\피로도_df_최종.csv")
output_path = input_path.parent / 'fatigue_df_with_consec_count.csv'

# 2) 데이터 읽기
df = pd.read_csv(input_path, encoding='cp949', parse_dates=['Date'])

# 3) 정렬
df = df.sort_values(['Name','Year','Date'])

# 4) 하루 차이 등판 여부 flag
df['is_consec'] = df.groupby(['Name','Year'])['Date'] \
                    .diff().dt.days.eq(1).astype(int)

# 5) flag 누적합 계산 (연투 발생 횟수 누적)
df['연투수'] = df.groupby(['Name','Year'])['is_consec'].cumsum()

# 6) 중간 flag 제거
df.drop(columns=['is_consec'], inplace=True)

# 7) '연투수'를 '연투일' 옆으로 이동
cols = df.columns.tolist()
if '연투수' in cols and '연투일' in cols:
    cols.remove('연투수')
    idx = cols.index('연투일') + 1
    cols.insert(idx, '연투수')
df = df[cols]

# 8) 결과 확인
print(df[['Name','Year','Date','연투일','연투수']].head(10))

# 9) 저장
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✔ 저장 완료: {output_path}")

  Name  Year       Date  연투일  연투수
0  고영표  2021 2021-04-07    1    0
1  고영표  2021 2021-04-13    1    0
2  고영표  2021 2021-04-18    1    0
3  고영표  2021 2021-04-24    1    0
4  고영표  2021 2021-04-30    1    0
5  고영표  2021 2021-05-06    1    0
6  고영표  2021 2021-05-12    1    0
7  고영표  2021 2021-05-26    1    0
8  고영표  2021 2021-06-01    1    0
9  고영표  2021 2021-06-08    1    0
✔ 저장 완료: C:\Users\yun72_92xubzr\Downloads\fatigue_df_with_consec_count.csv


In [16]:
import pandas as pd
from pathlib import Path

# 1) 파일 경로 설정
input_path  = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\최진원\학업\교외 활동\대외 활동\DArt-B\5기\학술제\크롤링\2025년.csv")
output_path = input_path.parent / 'fatigue_df.csv'

# 2) 데이터 읽기
df = pd.read_csv(input_path, encoding='cp949', parse_dates=['Date'])

# 3) 컬럼명 매핑 딕셔너리
rename_dict = {
    'Name': '이름', 'Year': '연도', 'Team': '팀', 'Date': '날짜', 'Role': '보직',
    'Venue': '구장', 'Temp': '온도',
    'ERA': 'ERA', 'WHIP': 'WHIP', 'IP': '이닝', 'NP': '투구수',
    'R': '실점', 'ER': '자책점', 'H': '피안타', 'HR': '피홈런', 'SO': '탈삼진', 'BB': '볼넷', 'HB': '사구',
    'FIP': 'FIP', 'delta_FIP': 'FIP_변화량', 'GSv2': 'GS', 'avg_GSv2': '평균_GS', 'delta_GSv2': 'GS_변화량',
    'Total_IP': '총_이닝', 'avg_FIP': '평균_FIP', 'avg_ERA': '평균_ERA', 'delta_ERA': 'ERA_변화량',
    '직구_피안타율': '직구_피안타율', '직구_구사율': '직구_구사율', '변화구_구사율': '변화구_구사율',
    '연투 여부': '연투여부', '휴식일 수': '휴식일수', '연투일': '연투일수', '연투수': '연투횟수',
    '2023WBC': 'WBC_2023', '2021올림픽': '올림픽_2021', '2023아시안게임': '아시안게임_2023',
    '2023아시아프로야구챔피언십': '아시아챔피언십_2023', '2023 아시아 야구 선수권 대회': '아시아선수권_2023',
    '2019프리미어12': '프리미어12_2019',
    'PS_KS': 'PS_KS', 'PS_PO': 'PS_PO', 'PS_SP': 'PS_SP', 'PS_V': 'PS_V', 'PS_WC': 'PS_WC',
    'Height': '키', 'Weight': '몸무게', 'Age': '나이',
    '총이동거리': '총이동거리', '이동거리': '이동거리', '누적이동거리': '누적이동거리', '누적_IP': '누적이닝',
    '속도': '구속', 'avg_속도': '평균구속', 'delta_속도': '구속변화량'
}

# 4) 컬럼명 변경
df.rename(columns=rename_dict, inplace=True)

# 5) 이름 + 날짜 기준 K/BB 계산용 서브 DataFrame 생성
kbb_df = df[['이름', '날짜', '탈삼진', '볼넷']].copy()
kbb_df['K_BB'] = kbb_df.apply(
    lambda row: row['탈삼진'] / row['볼넷'] if row['볼넷'] != 0 else float('inf'),
    axis=1
)

# 6) 기존 df에 병합 (이름 + 날짜 기준)
df = df.merge(kbb_df[['이름', '날짜', 'K_BB']], on=['이름', '날짜'], how='left')

# 7) 저장
df.to_csv(output_path, index=False, encoding='utf-8-sig')

# 8) 확인
print(df[['이름', '날짜', '탈삼진', '볼넷', 'K_BB']].head())

    이름         날짜  탈삼진  볼넷  K_BB
0  고영표 2025-03-25    5   1   5.0
1  고영표 2025-03-30    6   2   3.0
2  고영표 2025-04-08   10   0   inf
3  고영표 2025-04-15   11   2   5.5
4  고영표 2025-04-20    7   0   inf


In [7]:
# 1) 파일 경로 설정
input_path  = Path(r"C:\Users\yun72_92xubzr\Downloads\fatigue_df_with_KBB.csv")
output_path = input_path.parent / 'fatigue_df.csv'

# 2) 데이터 읽기
df = pd.read_csv(input_path, parse_dates=['날짜'])

ordered_columns = [
    # 1) 선수 기본 정보
    '이름', '연도', '팀', '보직', '나이', '키', '몸무게',

    # 2) 경기 정보
    '날짜', '구장', '온도',

    # 3) 투구 성적
    'ERA', 'WHIP',
    '이닝', '누적이닝', '총_이닝',
    '투구수', '실점', '자책점',
    '피안타', '피홈런', '탈삼진', '볼넷', '사구', 'K_BB',

    # 4) 연투 / 컨디션
    '연투여부', '휴식일수', '연투일수', '연투횟수',

    # 5) 국제 대회 경험
    'WBC_2023', '올림픽_2021', '아시안게임_2023',
    '아시아챔피언십_2023', '아시아선수권_2023', '프리미어12_2019',

    # 6) 포스트시즌 경험
    'PS_KS', 'PS_PO', 'PS_SP', 'PS_V', 'PS_WC',

    # 7) 이동 정보
    '이동거리', '총이동거리', '누적이동거리',

    # 8) 투구 스타일
    '직구_피안타율', '직구_구사율', '변화구_구사율',

    # 9) 고급 지표
    'FIP', 'FIP_변화량', '평균_FIP',
    'GS', '평균_GS', 'GS_변화량',
    '평균_ERA', 'ERA_변화량',
    '구속', '평균구속', '구속변화량'
]
# 2. 순서 적용
df = df[ordered_columns]

df.to_csv(output_path, index=False, encoding='utf-8-sig')

# 4. 확인
print(df.columns.tolist())


['이름', '연도', '팀', '보직', '나이', '키', '몸무게', '날짜', '구장', '온도', 'ERA', 'WHIP', '이닝', '누적이닝', '총_이닝', '투구수', '실점', '자책점', '피안타', '피홈런', '탈삼진', '볼넷', '사구', 'K_BB', '연투여부', '휴식일수', '연투일수', '연투횟수', 'WBC_2023', '올림픽_2021', '아시안게임_2023', '아시아챔피언십_2023', '아시아선수권_2023', '프리미어12_2019', 'PS_KS', 'PS_PO', 'PS_SP', 'PS_V', 'PS_WC', '이동거리', '총이동거리', '누적이동거리', '직구_피안타율', '직구_구사율', '변화구_구사율', 'FIP', 'FIP_변화량', '평균_FIP', 'GS', '평균_GS', 'GS_변화량', '평균_ERA', 'ERA_변화량', '구속', '평균구속', '구속변화량']
